# 03. Sorting

**Tier:** Intermediate. **Why you'd use this:** A leaderboard should list players by score, and when two scores match, keep the earlier name order you already sorted. You need `sorted` with a key, and you need that stability.

**After this lesson you can:** sort with `key=`; write a small `lambda` that picks the key; use two stable passes for a primary and secondary order.
About 25 minutes. **Uses:** [Lists](../04-collections/01-lists-lesson.ipynb), [Comprehensions](01-comprehensions-lesson.ipynb)


## Quick review

1. Why is a second loop over one generator empty?
2. What does `yield` do that `return` does not?

<details><summary>Answers</summary>

The generator is spent. `yield` pauses and can run again; `return` ends the call.
</details>


## The idea

`sorted` builds a new list in order. `key=` is a function that turns each item into the value used for comparisons. A `lambda` is a tiny one-expression function for that purpose. `sorted` is stable: items that compare equal keep their earlier order. Sort by the secondary key first, then by the primary key, to get a two-level order.

<details><summary>Explain more</summary>

`sorted(words, key=len)` compares lengths, not the strings. The original list is unchanged. There is no `cmp=` pairwise function anymore. Compose what you need with `key=` and stability.
</details>


## See it work

Order players by score, breaking ties with the name order from an earlier pass.


In [1]:
def leaderboard(rows):
    # 1. Secondary key first: name.
    by_name = sorted(rows, key=lambda row: row[0])
    # 2. Primary key second: score. Stability keeps the name order on ties.
    return sorted(by_name, key=lambda row: row[1])


print(leaderboard([("b", 2), ("a", 2), ("b", 1)]))

[('b', 1), ('a', 2), ('b', 2)]


## Check yourself

Answer these before you run the cell.

1. What order does `sorted(words, key=len)` produce?
2. After sorting by name then by score, which name comes first among score `2`?


In [2]:
words = ["aa", "b", "ccc"]
print(sorted(words, key=len))
rows = [("b", 2), ("a", 2), ("b", 1)]
by_name = sorted(rows, key=lambda row: row[0])
print(sorted(by_name, key=lambda row: row[1]))

['b', 'aa', 'ccc']
[('b', 1), ('a', 2), ('b', 2)]


<details><summary>Answers</summary>

Lengths give `['b', 'aa', 'ccc']`. Among score `2`, `"a"` stays before `"b"` because the name pass ran first and stability kept that order.
</details>


## Why it works this way

`sorted` copies the items into a new list and orders that copy. `key=` runs once per item and caches the result for comparisons, so an expensive key is not recomputed on every pairwise check. Stability means equal keys leave relative order alone. That is why a first pass by name, then a pass by score, yields score order with names broken alphabetically among ties. Without stability you would need one compound key that encodes both fields. With stability you compose two simple keys. Deep dive: [How list works](../../internals/03-lists/01-dynamic-array-lesson.ipynb).

`lambda` builds a small function object from one expression. It cannot contain statements, assignments, or annotations. For anything larger, write a `def` and pass that name as `key`. Old `cmp=` compared two items at a time and is gone. `key=` replaced it because decorating each item once is faster than asking a comparison function on every pair, and because most real sorts only need a projection of each item.

Cost is about n log n comparisons on the keyed values, plus one new list for `sorted`. Sorting in place with `list.sort` avoids the copy when you do not need the old order. Two stable passes cost two sorts. That is still the right tool when the keys are independent and easy to name. A single-pass key that returns a tuple of `(primary, secondary)` also works and is often clearer once you know the order of comparison on tuples.

The design goal is a predictable order from a simple projection. Reach for `key=` before you write a custom comparison class. Reach for a named function before you stuff a multiline idea into a `lambda`. Keep the original list when another part of the program still needs the input order.


In [3]:
words = ["aa", "b", "ccc", "dd"]
print(sorted(words, key=len))
rows = [("b", 2), ("a", 2), ("b", 1)]
by_name = sorted(rows, key=lambda row: row[0])
print(by_name)
print(sorted(by_name, key=lambda row: row[1]))

['b', 'aa', 'dd', 'ccc']
[('a', 2), ('b', 2), ('b', 1)]
[('b', 1), ('a', 2), ('b', 2)]


## When to use it

Use `sorted` when you need a new list. Use `list.sort` when the old order can go. Use `key=` for the comparison value. Use two stable passes for a primary and secondary key. Don't force a complex `lambda` with statements. Write a named function instead.


## Common mistake

Sorting only by the primary key and expecting ties to follow another field. Without the earlier pass, equal scores keep whatever order the input had, which may not be by name.


In [4]:
rows = [("b", 2), ("a", 2), ("b", 1)]
by_score = sorted(rows, key=lambda row: row[1])
print(by_score)

[('b', 1), ('b', 2), ('a', 2)]


Sort by name first, then by score. Stability keeps `"a"` before `"b"` at score `2`.


In [5]:
rows = [("b", 2), ("a", 2), ("b", 1)]
by_name = sorted(rows, key=lambda row: row[0])
print(sorted(by_name, key=lambda row: row[1]))

[('b', 1), ('a', 2), ('b', 2)]


## Try it

Change `leaderboard` so higher scores come first. Which argument do you add to the second `sorted`, and do you still need the name pass?


## Recap

- `sorted` returns a new list. `key=` chooses what to compare.
- `lambda` is a one-expression function for simple keys.
- Stability lets two passes build a primary and secondary order.

**Next:** [practice](03-sorting-practice.ipynb)
